# My glacier — mini-project notebook

This notebook is your **lab book** for the mini-project (see `project/project.md`). Run it from `project/my-glacier/`, in your RenkuLab session.

Steps:
0. Set your glacier in `experiment/common/my_glacier.yaml` (`RGI_ID`)
1. **Present**: download the data, check the initial state
2. **Past**: Little Ice Age → today (calibration)
3. **Future**: scenarios 2020–2100
4. Figures for the report

⚠️ Download your notebook, YAML files and figures regularly (right-click > Download).

In [ ]:
import sys, os, re, yaml
sys.path.insert(0, os.path.abspath("../../utils"))   # course plotting helpers
from helper import *

with open("experiment/common/my_glacier.yaml") as f:
    RGI_ID = yaml.safe_load(f)["inputs"]["oggm_shop"]["RGI_ID"]
INPUT = f"data/input_{RGI_ID}.nc"
print("My glacier:", RGI_ID)

## Run log

Keep track of every run here (double-click to edit):

| Run folder | Experiment file | What I changed | Why | Result |
|---|---|---|---|---|
| `outputs/check` | `params_future` | 1-year test run | download data | |
| | | | | |

## 1. Present: download the data and check the initial state

A 1-year run downloads the data (≈ 1 min the first time) and gives the modelled velocity field of today's glacier.

In [ ]:
!igm_run +experiment=params_future processes.time.end=2021 hydra.run.dir=outputs/check

In [ ]:
plot_input_data(INPUT)
plot_climate(RGI_ID)

In [ ]:
# Modelled (2020) vs observed surface speeds -> adjust tau_ref / arrhenius in common/my_glacier.yaml if needed
compare_modeled_observed_velocity(INPUT, "outputs/check/output.nc", year=2020)

**Notes** (initial state, observations, present-day climate offset printed above, ...):

- ...

## 2. Past: from the Little Ice Age to today

Start in 1700 from today's geometry, advance to the LIA maximum (~1850), retreat with the observed climate 1901–2019. Edit `experiment/params_past.yaml` (or a copy), run, look, iterate. A 320-year run takes a few minutes.

In [ ]:
!cat experiment/params_past.yaml

In [ ]:
!igm_run +experiment=params_past hydra.run.dir=outputs/past_v1

In [ ]:
run = "outputs/past_v1"
show_dashboard(run, n=4)
analyze_time_series(os.path.join(run, "output_ts.nc"))

In [ ]:
# Ice thickness through time, and along the main flowline
animate_glacier_evolution(os.path.join(run, "output.nc"), variable="thk")
plot_flowline_evolution(os.path.join(run, "output.nc"), rgi_id=RGI_ID)

In [ ]:
# Compare several past runs with today's observed area / volume (from the input data)
ds = xr.open_dataset(INPUT); dx = float(ds.x[1] - ds.x[0])
obs = {"area": ([2020], [float((ds.thk > 1).sum()) * dx**2 / 1e6]),
       "vol":  ([2020], [float(ds.thk.sum()) * dx**2 / 1e9])}
compare_runs(["outputs/past_v1"], ["past v1"], observations=obs)

✅ Is the LIA maximum reached around 1850? Does the front reach the LIA moraines (compare with the Siegfried map on map.geo.admin.ch)? Is today's extent right? If not, change `delta_temp` (before 1901) or `melt_enhancer`, save as a new file / run folder (`past_v2`, ...), and compare.

**Calibrated values**: `delta_temp` (pre-1901) = ..., `melt_enhancer` = ...

## 3. Future: scenarios 2020–2100

`delta_temp` is relative to **1960–1990** (see `project.md`):
- 2020 row: present-day offset (printed by `plot_climate`)
- 2100 row: 1981–2010 offset + CH2018 warming of the scenario

The cell below writes one experiment file per scenario (`experiment/params_future_<name>.yaml`) from `params_future.yaml`, with your calibrated `melt_enhancer`, then runs them all.

In [ ]:
MELT_ENHANCER = 1.0      # <- calibrated in Step 2
DT_2020 = 1.3            # <- present-day offset vs 1960-1990 (plot_climate)
DT_8110 = 0.6            # <- 1981-2010 offset vs 1960-1990 (plot_climate)

scenarios = {            # name: delta_temp in 2100 (vs 1960-1990)
    "committed": DT_2020,          # today's climate kept until 2100
    "rcp26":     DT_8110 + 1.2,
    "rcp45":     DT_8110 + 2.6,
    "rcp85":     DT_8110 + 4.3,
}

template = open("experiment/params_future.yaml").read()
for name, dt2100 in scenarios.items():
    txt = re.sub(r"- \[ 2020,\s*[-\d.]+,", f"- [ 2020, {DT_2020:5.2f},", template)
    txt = re.sub(r"- \[ 2100,\s*[-\d.]+,", f"- [ 2100, {dt2100:5.2f},", txt)
    txt = re.sub(r"melt_enhancer: [\d.]+", f"melt_enhancer: {MELT_ENHANCER}", txt)
    with open(f"experiment/params_future_{name}.yaml", "w") as f:
        f.write(txt.replace("# @package _global_", f"# @package _global_\n# Generated by my_glacier.ipynb: scenario {name}", 1))
    print(f"{name:10s}: delta_temp 2020 = {DT_2020:+.2f}, 2100 = {dt2100:+.2f} °C (vs 1960-1990)")

In [ ]:
for name in scenarios:
    !igm_run +experiment=params_future_{name} hydra.run.dir=outputs/future_{name}

In [ ]:
compare_runs([f"outputs/future_{n}" for n in scenarios], list(scenarios))

In [ ]:
# Maps for one scenario
animate_glacier_evolution("outputs/future_rcp45/output.nc", variable="thk")

## 4. Figures for the report

The helper functions in `utils/helper.py` are meant for quick looks. For the report, make your own figures: copy the code of a helper into a cell below and adapt it (years shown, titles, units, colors, fonts), then save with `plt.savefig("fig_xxx.png", dpi=200, bbox_inches="tight")` **before** `plt.show()`. (To grab a quick-look figure, right-click > *Save image as*.)

In [ ]:
# your figures here